# Модуль 1 · Урок 4. NumPy: операции, axis и broadcasting
**AI-Driven ML Engineer · Data Science Academy**

| Время | Блок |
|---|---|
| 0–10 | Разминка «что выведет код» |
| 10–25 | Ответы ДЗ 3, устная проверка |
| 25–40 | Арифметика с массивами и функции NumPy |
| 40–60 | Статистики и axis · Практика 1 |
| 60–70 | Перерыв |
| 70–90 | Broadcasting · Практика 2 |
| 90–110 | Практика 3: таблица «город × месяц» по orders.csv |
| 110–120 | Итоги, ДЗ 4 |

**Правило занятия:** предскажи → запусти → объясни.

## Разминка: что выведет код? (30 секунд на вопрос, без запуска)
```python
import numpy as np
# 1
a = np.array([1, 2, 3]); print(a * 2)
# 2
print([1, 2, 3] * 2)
# 3
m = np.arange(6).reshape(2, 3); print(m.sum(axis=0))
# 4
print(np.array([1, 0]) / np.array([0, 0]))
# 5
print(np.array([[1], [2]]) + np.array([10, 20]))
```

In [ ]:
# Проверьте ответы только ПОСЛЕ того, как написали их в чат

## Ответы ДЗ 3 (кратко)
- Самый крупный заказ: позиция 1001, 9 999 990 ₸, клиент TEST, Алматы.
- Оплачено 101 909 360 ₸, не оплачено 7 974 000 ₸.
- Шымкент: 804 заказа, 14,7% всех заказов. За 2 дня там доставляют 30,7% заказов, в других городах 62,8–71,0%.
- Скидка 15%: 5 702 422 ₸ с TEST и 1 202 427 ₸ без TEST. Три тестовых заказа дают почти 80% суммы.
- Средний чек `amount.mean()` = 20 114 ₸ неверный: в нём TEST и отмены. Без них 14 672 ₸ (дубли ещё внутри, их уберём сегодня).

## 1. Арифметика с массивами (03-Operations)
Операция с массивом применяется к каждому элементу. Два массива одной формы складываются поэлементно.

In [1]:
import numpy as np
arr = np.arange(0, 10)
print(arr + arr)
print(arr * arr)
print(arr - arr)
print(arr * 2)
print(arr ** 2)

[ 0  2  4  6  8 10 12 14 16 18]
[ 0  1  4  9 16 25 36 49 64 81]
[0 0 0 0 0 0 0 0 0 0]
[ 0  2  4  6  8 10 12 14 16 18]
[ 0  1  4  9 16 25 36 49 64 81]


In [2]:
# ПРЕДСКАЖИТЕ: список и массив ведут себя по-разному
lst = [1, 2, 3]
print(lst * 2)             # список: повторить
print(np.array(lst) * 2)   # массив: умножить каждый элемент

[1, 2, 3, 1, 2, 3]
[2 4 6]


In [6]:
# Деление на ноль не останавливает программу: только предупреждение
print(arr / arr)   # 0/0 = nan (not a number)

print(1 / arr)     # 1/0 = inf (бесконечность)
arr / arr

[nan  1.  1.  1.  1.  1.  1.  1.  1.  1.]
[       inf 1.         0.5        0.33333333 0.25       0.2
 0.16666667 0.14285714 0.125      0.11111111]


/var/folders/3v/3zf00xcj57z36ph93p9n4qk80000gn/T/ipykernel_25154/1614734774.py:2: RuntimeWarning: invalid value encountered in divide
  print(arr / arr)   # 0/0 = nan (not a number)
/var/folders/3v/3zf00xcj57z36ph93p9n4qk80000gn/T/ipykernel_25154/1614734774.py:4: RuntimeWarning: divide by zero encountered in divide
  print(1 / arr)     # 1/0 = inf (бесконечность)
/var/folders/3v/3zf00xcj57z36ph93p9n4qk80000gn/T/ipykernel_25154/1614734774.py:5: RuntimeWarning: invalid value encountered in divide
  arr / arr


array([nan,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.])

> `nan` мы уже видели в `delivery_days`: пустая клетка таблицы. Любая арифметика с `nan` даёт `nan`, поэтому `days.mean()` = nan.

### Функции NumPy (universal functions)
Работают с каждым элементом: `np.sqrt`, `np.exp`, `np.log`, `np.abs`, `np.round`.

In [13]:
print(np.sqrt(arr))
print(np.round(np.log(arr[1:]), 2))

# Бизнес-пример: суммы заказов в тысячах тенге, одна строка вместо цикла
prices = np.array([15990.0, 4500.0, 89000.0])
print(np.round(prices / 1000, 1))

np.round(prices / 1000, 2)
#np.round(prices / 1000)

[0.         1.         1.41421356 1.73205081 2.         2.23606798
 2.44948974 2.64575131 2.82842712 3.        ]
[0.   0.69 1.1  1.39 1.61 1.79 1.95 2.08 2.2 ]
[16.   4.5 89. ]


array([15.99,  4.5 , 89.  ])

## 2. Статистики и axis
`sum`, `mean`, `std`, `min`, `max`, `argmax` считают по всему массиву. С параметром `axis` считают по одному измерению.

```
axis=0: идём ВНИЗ по строкам  → получаем результат для каждого СТОЛБЦА
axis=1: идём ВПРАВО по столбцам → получаем результат для каждой СТРОКИ

        столбцы →
      [[ 1  2  3]      sum(axis=1) → [ 6
       [ 4  5  6]]                     15]
  sum(axis=0) → [5 7 9]
```

In [16]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m.sum())          # 21
print(m.sum(axis=0))    # по столбцам: [5 7 9]
print(m.sum(axis=1))    # по строкам:  [ 6 15]
print(m.max(axis=0), m.max(axis=1))
m.mean()

21
[5 7 9]
[ 6 15]
[4 5 6] [3 6]


3.5

## Практика 1 (вторая половина 04-Numpy Exercises), 10 минут
```python
mat = np.arange(1, 26).reshape(5, 5)
```
1. **[База]** Сумма всех элементов.
2. **[База]** Стандартное отклонение всех элементов.
3. **[База]** Сумма каждого столбца.
4. **[Стандарт]** Сумма каждой строки и среднее каждого столбца.
5. **[Челлендж]** Максимум в каждой строке и номер столбца, где он стоит (`argmax` с axis). Почему номер везде одинаковый?

In [17]:
mat = np.arange(1, 26).reshape(5, 5)
mat

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25]])

In [19]:
# 1
mat.sum()

325

In [20]:
# 2
mat.std()

7.211102550927978

In [21]:
# 3
mat.sum(axis=0)

array([55, 60, 65, 70, 75])

In [22]:
# 4
mat.sum(axis=1)

array([ 15,  40,  65,  90, 115])

In [24]:
mat.mean(axis=0)

array([11., 12., 13., 14., 15.])

In [28]:
mat

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25]])

In [27]:
mat.max(axis=1)

array([ 5, 10, 15, 20, 25])

In [29]:
# 5
mat.max(axis=1).argmax()

4

In [31]:
mat[:,2:3] = 50
mat

array([[ 1,  2, 50,  4,  5],
       [ 6,  7, 50,  9, 10],
       [11, 12, 50, 14, 15],
       [16, 17, 50, 19, 20],
       [21, 22, 50, 24, 25]])

In [33]:
mat.max(axis=1).argmax()

0

In [34]:
mat.argmax(axis=1)

array([2, 2, 2, 2, 2])

## Перерыв 10 минут

## 3. Broadcasting
Когда формы массивов разные, NumPy пытается «растянуть» меньший массив.

**Правило:** формы сравниваются справа налево. Размеры совпадают или один из них равен 1. Размер 1 растягивается.

| Форма A | Форма B | Результат |
|---|---|---|
| (3, 3) | (3,) | (3, 3): строка прибавляется к каждой строке |
| (3, 3) | (3, 1) | (3, 3): столбец прибавляется к каждому столбцу |
| (2, 1) | (2,) | (2, 2): растягиваются оба |
| (6, 2) | (6,) | ошибка: 2 ≠ 6 |

In [35]:
np.zeros((3, 3))

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [36]:
m = np.zeros((3, 3))
print(m + np.array([1, 2, 3]))          # строка (3,) растянута на 3 строки
print(m + np.array([[1], [2], [3]]))    # столбец (3, 1) растянут на 3 столбца

[[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]
[[1. 1. 1.]
 [2. 2. 2.]
 [3. 3. 3.]]


In [41]:
t = np.ones((6, 2))
t

array([[1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.]])

In [40]:
t.sum(axis=1)

array([2., 2., 2., 2., 2., 2.])

In [42]:
t.sum(axis=1).shape

(6,)

In [43]:
# ПРЕДСКАЖИТЕ: сработает ли?
t = np.ones((6, 2))
try:
    print(t / t.sum(axis=1))
except ValueError as e:
    print("Ошибка:", e)

print((t / t.sum(axis=1, keepdims=True))[:2])   # keepdims оставляет форму (6, 1)

Ошибка: operands could not be broadcast together with shapes (6,2) (6,) 
[[0.5 0.5]
 [0.5 0.5]]


## Практика 2: продажи трёх товаров за два дня, 10 минут
```python
prices = np.array([15990, 4500, 89000])   # цена: наушники, книга, телефон
qty = np.array([[2, 10, 1],               # день 1: штук каждого товара
                [3,  8, 0]])              # день 2
```
1. **[База]** Выручка по каждому товару в каждый день: `qty * prices`. Какая форма у результата и почему это broadcasting?
2. **[База]** Выручка за каждый день и по каждому товару за два дня (`axis`).
3. **[Стандарт]** Доля каждого товара в выручке своего дня. Сумма долей в строке должна быть 1.
4. **[Челлендж]** Цена выросла на 10% только у телефона. Пересчитайте выручку одним выражением с вектором коэффициентов.

In [44]:
prices = np.array([15990, 4500, 89000])
qty = np.array([[2, 10, 1],
                [3,  8, 0]])

In [45]:
prices.shape

(3,)

In [46]:
qty.shape

(2, 3)

In [49]:
# 1
revenue = prices * qty

In [48]:
qty * prices

array([[31980, 45000, 89000],
       [47970, 36000,     0]])

In [51]:
# 2
by_day = revenue.sum(axis = 1)
by_day

array([165980,  83970])

In [52]:
by_item = revenue.sum(axis = 0)
by_item

array([79950, 81000, 89000])

In [55]:
revenue

array([[31980, 45000, 89000],
       [47970, 36000,     0]])

In [54]:
# 3
revenue / revenue.sum(axis=1, keepdims=True)

array([[0.19267382, 0.271117  , 0.53620918],
       [0.57127546, 0.42872454, 0.        ]])

In [59]:
# 4 Цена выросла на 10% только у телефона. 
# Пересчитайте выручку одним выражением с вектором коэффициентов.

coef = np.array([1, 1, 1.1])
print(prices * coef)

revenue_new = qty * (prices * coef)
revenue_new


[15990.  4500. 97900.]


array([[31980., 45000., 97900.],
       [47970., 36000.,     0.]])

## 4. orders.csv: таблица «город × месяц»
Вернёмся к вопросу из ДЗ 2: почему выручка в октябре упала? Сегодня соберём таблицу выручки по городам и месяцам и ответим на него операциями с массивами.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

In [ ]:
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

## Практика 3, 20 минут
1. **[База]** Выручка за каждый месяц (`axis`) и изменение октября к сентябрю в %. Должно совпасть с ДЗ 2: около −2,3%.
2. **[База]** Изменение в % для каждого города одной строкой: `table[:, 1] / table[:, 0] - 1`. Где падение сильнее всего?
3. **[Стандарт]** Вклад каждого города в общее падение: изменение города / общее изменение. Какую долю падения даёт Шымкент?
4. **[Стандарт]** Доля каждого города в выручке месяца: `table / table.sum(axis=0)`. Проверьте, что каждый столбец в сумме даёт 1.
5. **[Челлендж]** Доля сентября и октября внутри каждого города. Почему `table / table.sum(axis=1)` даёт ошибку и как её исправить?

In [ ]:
# 1

In [ ]:
# 2

In [ ]:
# 3

In [ ]:
# 4

In [ ]:
# 5

## Итоги

| Вопрос | Инструмент |
|---|---|
| Посчитать для каждого элемента | `arr * 2`, `np.sqrt(arr)` |
| Сумма по городам или по месяцам | `table.sum(axis=1)`, `table.sum(axis=0)` |
| Поделить таблицу на итоги | broadcasting: `table / table.sum(axis=0)` |
| Итоги по строкам с делением | `keepdims=True` |
| Убрать дубли | `np.unique(oid, return_index=True)` |

**ДЗ 4:** ноутбук HW04_numpy, сдача в папку `hw04` репозитория `dsa-homework`.